# Titanic EDA

Goal: understand the strongest survival signals before we trust the model. Every chart below answers a specific question. If the chart shows a clear pattern, the model should be able to learn it — and feature importances later should reflect it. If a chart shows no pattern, that feature is probably noise.

Survival overall was 38.4%, so any subgroup well above or below that line is informative.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make src/ importable so we can reuse the title extractor.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from pipeline import extract_title  # noqa: E402

sns.set_theme(style="whitegrid")
train = pd.read_csv(PROJECT_ROOT / "data" / "train.csv")
print(f"{len(train)} rows, {train['Survived'].mean():.1%} survival rate")
train.head()

## 1. Missing values

Before anything else: which columns have gaps? Missingness drives our imputation strategy. A column that's 77% missing (like Cabin) gets a totally different treatment from one that's 0.2% missing (like Embarked).

In [ ]:
missing = train.isna().sum()
missing_pct = (missing / len(train) * 100).round(1)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

**Takeaway.** Cabin is mostly missing (we'll extract Deck + treat 'NoCabin' as its own category). Age is ~20% missing (we'll impute by Title+Pclass median). Embarked is trivially missing (mode-fill is fine).

## 2. Survival by Sex × Pclass

The single most important pattern in this dataset. "Women and children first" was the lifeboat policy, and class drove access to the upper decks.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=train, x="Pclass", y="Survived", hue="Sex", ax=ax)
ax.axhline(0.384, color="gray", linestyle="--", label="overall rate")
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by class and sex")
ax.legend()
plt.show()

train.groupby(["Sex", "Pclass"])["Survived"].mean().unstack().round(3)

**Takeaway.** A 1st-class woman had a ~97% survival rate; a 3rd-class man, ~14%. This is the dominant signal — any model that doesn't strongly weight Sex × Pclass is broken. Notice that 3rd-class women (50%) survived at a much lower rate than 1st/2nd-class women (~95%) — the class effect is real even among women.

## 3. Survival by Title

Title (Mr, Mrs, Miss, Master) is a powerful feature because it bundles sex, marital status, and (for Master) age. Master = boys under ~12, where the 'children first' policy kicks in.

In [ ]:
train["Title"] = train["Name"].apply(extract_title)
title_summary = (
    train.groupby("Title")
    .agg(n=("Survived", "size"), survival_rate=("Survived", "mean"))
    .sort_values("survival_rate", ascending=False)
    .round(3)
)
print(title_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=train, x="Title", y="Survived", ax=ax,
            order=title_summary.index)
ax.axhline(0.384, color="gray", linestyle="--")
ax.set_title("Survival rate by Title")
plt.show()

**Takeaway.** Mrs (~79%) and Miss (~70%) survived at much higher rates than Mr (~16%). Master — boys — clocks in at ~57%, evidence that the 'children first' rule had teeth. 'Rare' (Don, Lady, Capt, etc.) is mostly upper-class adults with mixed outcomes.

## 4. Survival vs. Family Size

Family size is nonlinear: solo travelers and big families both fared worse than small groups of 2–4.

In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
fs_summary = (
    train.groupby("FamilySize")
    .agg(n=("Survived", "size"), survival_rate=("Survived", "mean"))
    .round(3)
)
print(fs_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=train, x="FamilySize", y="Survived", ax=ax)
ax.axhline(0.384, color="gray", linestyle="--")
ax.set_title("Survival rate by family size (SibSp + Parch + 1)")
plt.show()

**Takeaway.** Solo travelers (FamilySize=1, the largest group) had ~30% survival — below average. Families of 2–4 did much better (50–72%). Families of 5+ collapsed to ~16–0% — they couldn't coordinate getting to lifeboats together. This non-linearity is why FamilySize works better than SibSp and Parch individually.

## 5. Age distribution by survival

Are children really more likely to have survived? Plot Age distributions split by outcome.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, "crimson"), (1, "seagreen")]:
    subset = train[train["Survived"] == label]["Age"].dropna()
    ax.hist(subset, bins=30, alpha=0.5, color=color,
            label=f"Survived={label}")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.set_title("Age distribution by survival")
ax.legend()
plt.show()

**Takeaway.** Look at ages 0–10 — green clearly dominates, confirming the children-survived effect. For adults the two distributions overlap a lot, meaning Age alone is a weak signal in the adult range. The 60+ tail is sparse — small sample, don't over-interpret.

## 6. Fare distribution and survival

Fare is a proxy for class but with more resolution. It's heavily right-skewed — a handful of passengers paid 200+ while the median is under 15. We'll log-scale the x-axis to see the shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, "crimson"), (1, "seagreen")]:
    subset = train[train["Survived"] == label]["Fare"]
    # +1 to handle the few zero-fare passengers (free tickets / crew?)
    ax.hist(np.log1p(subset), bins=30, alpha=0.5, color=color,
            label=f"Survived={label}")
ax.set_xlabel("log(Fare + 1)")
ax.set_ylabel("Count")
ax.set_title("Fare distribution by survival (log scale)")
ax.legend()
plt.show()

print(train.groupby("Survived")["Fare"].describe().round(2))

**Takeaway.** Survivors paid substantially more on average ($48 mean vs. $22). The distributions overlap but are clearly shifted. This is largely a Pclass restatement, but Fare has more resolution within each class.

## Putting it together

Strongest signals (in roughly this order):
1. **Sex × Pclass** — captures the women-and-children-first × class-access interaction.
2. **Title** — augments Sex by marking children (Master) and marital status.
3. **FamilySize** — nonlinear: small groups good, big groups bad.
4. **Fare / FarePerPerson** — finer-grained class proxy.
5. **Age** — weak in adults, strong only for children.

When we look at the Random Forest's feature importances in `outputs/cv_scores.txt`, we expect the top of the list to mirror this story.